# 4.01 PDF Document Loaders

PDF는 문서 로더 중 가장 자주 쓰이는 포맷이지만, 내부 구조가 파서마다 다르게 해석되어 **표·이미지·레이아웃 품질** 편차가 크다.
이 노트북은 LangChain에 탑재된 5종의 PDF 로더를 같은 파일에 돌려 품질과 속도를 비교하고, 디렉터리 단위 적재와 RAG 파이프라인 연결을 다룬다.

## 학습 목표

- `PyPDFLoader` · `PyMuPDFLoader` · `PyMuPDF4LLMLoader`(직접 호출 경로) · `PDFPlumberLoader` · `PDFMinerLoader`의 출력 형태를 비교한다
- 표·레이아웃 보존에 강한 파서와 속도 우선 파서를 구분한다
- `PyPDFDirectoryLoader`로 폴더 단위 대량 적재를 수행한다
- PDF → `RecursiveCharacterTextSplitter` → `Chroma` 흐름으로 RAG 파이프라인을 한 줄로 연결한다

## 언제 쓰나

- 스캔 없이 텍스트 레이어가 살아있는 PDF(대부분의 리포트·논문)
- 표가 많은 재무/운영 자료 → `PyMuPDFLoader(extract_tables=...)`, `PDFPlumberLoader`
- LLM에 바로 먹일 **markdown** 품질이 필요할 때 → `pymupdf4llm.to_markdown(...)`
- 수십~수백 개 PDF 폴더 → `PyPDFDirectoryLoader`

## 4.01.1 환경 설정

필요 패키지: `pypdf`, `pymupdf`, `pymupdf4llm`, `pdfplumber`, `pdfminer.six`, `reportlab`(샘플 PDF 생성용), `langchain-community`.

아래 probe는 실습용 PDF를 현재 디렉터리에 직접 생성한다. 이후 모든 셀은 이 파일을 공유 입력으로 사용한다.

In [ ]:
# !pip install -U pypdf pymupdf pymupdf4llm pdfplumber pdfminer.six reportlab langchain-community
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(override=True)

from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

SAMPLE_DIR = Path("./_pdf_samples").resolve()
SAMPLE_DIR.mkdir(exist_ok=True)
SAMPLE_PDF = SAMPLE_DIR / "report.pdf"

styles = getSampleStyleSheet()
doc = SimpleDocTemplate(str(SAMPLE_PDF), pagesize=A4)
story = [
    Paragraph("LangChain PDF Loader Sample", styles["Title"]),
    Spacer(1, 12),
    Paragraph("Document loaders convert raw PDFs into LangChain Document objects.", styles["BodyText"]),
    Spacer(1, 12),
    Paragraph("Quarterly revenue snapshot:", styles["Heading2"]),
    Table(
        [["Quarter", "Revenue", "Growth"],
         ["Q1", "1,200", "+5%"],
         ["Q2", "1,450", "+21%"],
         ["Q3", "1,610", "+11%"]],
        style=TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
            ("GRID", (0, 0), (-1, -1), 0.5, colors.black),
        ]),
    ),
    Spacer(1, 12),
    Paragraph("Page two follows with additional narrative for multi-page tests.", styles["BodyText"]),
]
doc.build(story + [Paragraph("Appendix A", styles["Heading2"]),
                   Paragraph("Second-page body text used to verify page-level splitting.", styles["BodyText"])])

# 두 번째 파일: 디렉터리 로더용
SECOND_PDF = SAMPLE_DIR / "memo.pdf"
c = canvas.Canvas(str(SECOND_PDF), pagesize=A4)
c.drawString(72, 800, "Internal memo — loader directory test")
c.drawString(72, 780, "This file exists only so PyPDFDirectoryLoader sees > 1 PDF.")
c.showPage()
c.save()

assert SAMPLE_PDF.exists() and SECOND_PDF.exists(), "샘플 PDF 생성 실패"
print("sample:", SAMPLE_PDF)
print("bytes :", SAMPLE_PDF.stat().st_size)

## 4.01.2 기본 로드 — `PyPDFLoader`

`PyPDFLoader(file_path, mode="page")`는 페이지당 하나의 `Document`를 반환한다(`mode="single"`이면 전체 합본 1개).
`pypdf` 기반으로 가볍고 의존성이 적어 **기본값으로 가장 많이 쓰인다**.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(SAMPLE_PDF))  # 기본 mode='page'
docs = loader.load()
print("문서 수:", len(docs))
for d in docs[:2]:
    print("---")
    print("metadata:", {k: d.metadata[k] for k in list(d.metadata)[:6]})
    print("text[:120]:", d.page_content[:120].replace("\n", " "))

## 4.01.3 파서 비교 — 속도 · 표 · markdown 품질

같은 PDF를 다섯 가지 경로로 로드해 **문자 수**와 **실행 시간**을 나란히 본다.

| 파서 | 강점 | 약점 | 비고 |
|------|------|------|------|
| `PyPDFLoader` (pypdf) | 가볍고 빠름, 설치 안전 | 표 보존 약함 | 기본값 |
| `PyMuPDFLoader` (pymupdf) | 빠르고 표(`extract_tables=...`) 지원 | AGPL/Commercial 라이선스 주의 | 속도 우선 |
| `pymupdf4llm.to_markdown` | 표·레이아웃을 markdown으로 변환, LLM 친화 | 직접 호출(LC 커뮤니티 로더 미등록) | markdown 품질 최상 |
| `PDFPlumberLoader` | 표 추출에 강함 | 느림 | 표가 많은 문서 |
| `PDFMinerLoader` | 레이아웃 좌표 활용 | 가장 느림, 표 약함 | 레이아웃 분석

In [ ]:
import time
from langchain_community.document_loaders import PyMuPDFLoader, PDFPlumberLoader, PDFMinerLoader
import pymupdf4llm

def bench(name, fn):
    t0 = time.perf_counter()
    out = fn()
    elapsed = (time.perf_counter() - t0) * 1000
    text = "\n".join(d.page_content for d in out) if isinstance(out, list) else out
    return {"loader": name, "ms": round(elapsed, 1), "chars": len(text), "preview": text[:60].replace("\n", " ")}

results = [
    bench("PyPDFLoader      ", lambda: PyPDFLoader(str(SAMPLE_PDF)).load()),
    bench("PyMuPDFLoader    ", lambda: PyMuPDFLoader(str(SAMPLE_PDF)).load()),
    bench("PyMuPDFLoader+tbl", lambda: PyMuPDFLoader(str(SAMPLE_PDF), extract_tables="markdown").load()),
    bench("PDFPlumberLoader ", lambda: PDFPlumberLoader(str(SAMPLE_PDF)).load()),
    bench("PDFMinerLoader   ", lambda: PDFMinerLoader(str(SAMPLE_PDF)).load()),
    bench("pymupdf4llm.md   ", lambda: pymupdf4llm.to_markdown(str(SAMPLE_PDF))),
]
for r in results:
    print(f"{r['loader']} | {r['ms']:>6.1f} ms | chars={r['chars']:>5} | {r['preview']}")

### 표(Table) 보존 비교

같은 표를 각 파서가 어떻게 텍스트로 펼치는지 확인한다. `PyMuPDFLoader(extract_tables="markdown")`와 `pymupdf4llm.to_markdown`이 파이프 표 형식을 유지해 LLM이 가장 잘 읽는다.

In [ ]:
mu_tbl = PyMuPDFLoader(str(SAMPLE_PDF), extract_tables="markdown").load()
md_raw = pymupdf4llm.to_markdown(str(SAMPLE_PDF))
plain  = PyPDFLoader(str(SAMPLE_PDF)).load()

def section(title, body):
    print(f"\n===== {title} =====")
    print(body[:400])

section("PyMuPDFLoader(extract_tables='markdown') — page 1", mu_tbl[0].page_content)
section("pymupdf4llm.to_markdown (전체)", md_raw)
section("PyPDFLoader 기본 출력 — page 1", plain[0].page_content)

## 4.01.4 디렉터리 단위 적재 · `lazy_load`

`PyPDFDirectoryLoader(path, glob="**/[!.]*.pdf", recursive=...)`는 지정 폴더의 모든 PDF를 `PyPDFLoader`로 순회한다.
메모리에 전부 올리지 않으려면 `load()` 대신 `lazy_load()` 제너레이터를 쓰면 된다 — 대량 PDF에 필수 패턴.

`load()`는 내부에서 `lazy_load()`를 전부 소비한 리스트를 반환하므로, 수백 건 이상을 다룰 때는 제너레이터 경로를 권장한다.

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

dir_loader = PyPDFDirectoryLoader(str(SAMPLE_DIR), recursive=False)

# lazy_load: 한 번에 하나씩 yield, 메모리에 전체를 쌓지 않음
pages = []
for d in dir_loader.lazy_load():
    pages.append({
        "source": Path(d.metadata["source"]).name,
        "page": d.metadata.get("page"),
        "chars": len(d.page_content),
    })
for p in pages:
    print(p)

## 4.01.5 split · embed 연동 — 한 줄 RAG

PDF 로더의 출력은 `Document` 리스트이므로 `RecursiveCharacterTextSplitter` → `Chroma.from_documents` 한 줄로 검색기에 올릴 수 있다.
자세한 split 전략은 `../06_text_splitters/`, 벡터스토어 옵션은 `../03_vectorstores/`에서 다룬다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

assert os.environ.get("OPENAI_API_KEY"), "Chroma 임베딩용 OPENAI_API_KEY 필요"

raw_docs = PyMuPDFLoader(str(SAMPLE_PDF), extract_tables="markdown").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40).split_documents(raw_docs)

store = Chroma.from_documents(
    chunks,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name="pdf_demo",
)
for hit in store.similarity_search("quarterly revenue growth", k=2):
    print("-", hit.page_content[:120].replace("\n", " "))

## 체크리스트

- [ ] 기본값은 `PyPDFLoader`. 표가 중요하면 `PyMuPDFLoader(extract_tables="markdown")` 또는 `pymupdf4llm.to_markdown`.
- [ ] `mode="page"` vs `mode="single"` 선택에 따라 이후 split 전략이 달라진다.
- [ ] 수백 건 이상은 `load()` 대신 `lazy_load()` 제너레이터로 처리.
- [ ] PyMuPDF 계열은 AGPL 라이선스이므로 상용 배포 시 확인.

## 다음

- `02_web_crawl.ipynb` — 웹 문서/사이트맵 로더
- `../06_text_splitters/` — 로더 출력 → 청크 분할 전략

## 참고

- 공식 PDF 로더 가이드: https://python.langchain.com/docs/how_to/document_loader_pdf/
- PyMuPDF4LLM 문서: https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/
- pdfplumber 저장소: https://github.com/jsvine/pdfplumber